# 错误处理与调试

生产环境中 Agent 可能遇到各种错误。系统性的错误处理策略能提升可靠性。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

## 常见错误类型

| 错误 | 原因 | 解决 |
| --- | --- | --- |
| ModuleNotFoundError | 缺少依赖 | pip install |
| BadRequestError | API 参数错误 | 检查模型和参数 |
| RateLimitError | 频率限制 | 降低请求频率或增加重试 |
| TimeoutError | 请求超时 | 增加 timeout |
| ToolException | 工具执行异常 | 检查工具参数和逻辑 |

## 重试与降级

通过 `@wrap_model_call` 实现重试和降级。

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

@wrap_model_call
def safe_model_call(request, handler):
    """安全的模型调用（重试 + 友好提示）"""
    try:
        return handler(request)
    except Exception as e:
        print(f"模型调用异常: {e}")
        return handler(request)  # 重试一次

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model, middleware=[safe_model_call],
    system_prompt="你是菜鸟教程的助手。",
)

# result = agent.invoke({"messages": [HumanMessage(content="介绍一下菜鸟教程")]})
# print(result["messages"][-1].content[:80])

print("带错误处理的 Agent 已创建")

## 调试技巧

- `agent.get_graph().draw_mermaid()` 查看 Agent 流程图
- `stream_mode="debug"` 查看详细执行日志
- `@after_agent` 输出统计信息

**术语：**

- **重试**：失败后重新尝试
- **降级**：切换到备用服务
- **ToolException**：工具引发的异常，会被 Agent 捕获